# TP2 : Classification d'images médicales (Pneumonie) avec un CNN en PyTorch

**Objectifs :**
- Implémenter un CNN simple from scratch avec PyTorch pour détecter des pneumonies sur des radiographies thoraciques (classification binaire : Sain vs Pneumonie)
- Expérimenter avec différents hyperparamètres
- Analyser les performances du modèle

In [ ]:
# Installation des dépendances (Google Colab)
!pip install medmnist -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import medmnist
from medmnist import PneumoniaMNIST
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"MedMNIST version: {medmnist.__version__}")

## 1. Chargement du Dataset PneumoniaMNIST

In [ ]:
# Transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# Chargement des splits
train_dataset = PneumoniaMNIST(split='train', transform=transform, download=True)
val_dataset   = PneumoniaMNIST(split='val',   transform=transform, download=True)
test_dataset  = PneumoniaMNIST(split='test',  transform=transform, download=True)

BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_dataset)} images")
print(f"Val:   {len(val_dataset)} images")
print(f"Test:  {len(test_dataset)} images")

In [ ]:
# Visualisation d'exemples
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
labels_map = {0: 'Normal', 1: 'Pneumonie'}
for i in range(8):
    # Normal
    idx = np.where(train_dataset.labels.flatten() == 0)[0][i]
    img, lbl = train_dataset[idx]
    axes[0, i].imshow(img.squeeze(), cmap='gray')
    axes[0, i].set_title(labels_map[lbl.item()])
    axes[0, i].axis('off')
    # Pneumonie
    idx = np.where(train_dataset.labels.flatten() == 1)[0][i]
    img, lbl = train_dataset[idx]
    axes[1, i].imshow(img.squeeze(), cmap='gray')
    axes[1, i].set_title(labels_map[lbl.item()])
    axes[1, i].axis('off')
plt.suptitle("Exemples du dataset PneumoniaMNIST", fontsize=14)
plt.tight_layout()
plt.show()

## 2. Construction du Modèle CNN (2 blocs convolutionnels)

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Bloc 1 : Conv2d(1->16, 3x3) + ReLU + MaxPool2d(2)
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)  # 28x28 -> 14x14
        )
        # Bloc 2 : Conv2d(16->32, 3x3) + ReLU + MaxPool2d(2)
        self.block2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)  # 14x14 -> 7x7
        )
        # Couche FC : Flatten + Linear(32*7*7 -> 1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 1)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.classifier(x)
        return x

model = SimpleCNN().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nNombre total de paramètres: {total_params:,}")

## 3. Fonctions d'entraînement et d'évaluation

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device).float()
        optimizer.zero_grad()
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device).float()
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            preds = (torch.sigmoid(outputs) >= 0.5).float()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            total += labels.size(0)
    acc = accuracy_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    return total_loss / total, acc, recall, np.array(all_preds), np.array(all_labels)

In [ ]:
def run_experiment(model, train_loader, val_loader, test_loader, lr=0.001, epochs=5, optimizer_class=optim.Adam, device=device):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optimizer_class(model.parameters(), lr=lr)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, val_recall, _, _ = evaluate(model, val_loader, criterion, device)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} Recall: {val_recall:.4f}")

    # Evaluation finale sur le test set
    test_loss, test_acc, test_recall, preds, labels = evaluate(model, test_loader, criterion, device)
    print(f"\n=== Test Final === Accuracy: {test_acc:.4f} | Recall (Sensibilité): {test_recall:.4f}")
    print("\nRapport de classification:")
    print(classification_report(labels, preds, target_names=['Normal', 'Pneumonie']))
    print("Matrice de confusion:")
    print(confusion_matrix(labels, preds))
    return history, test_acc, test_recall

In [ ]:
def plot_history(history, title=""):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history['train_loss'], label='Train')
    ax1.plot(history['val_loss'], label='Validation')
    ax1.set_title(f'Loss {title}'); ax1.set_xlabel('Epoch'); ax1.legend()
    ax2.plot(history['train_acc'], label='Train')
    ax2.plot(history['val_acc'], label='Validation')
    ax2.set_title(f'Accuracy {title}'); ax2.set_xlabel('Epoch'); ax2.legend()
    plt.tight_layout(); plt.show()

## 3.1 Entraînement de base (2 blocs, lr=0.001, epochs=5, batch=32)

In [ ]:
model_base = SimpleCNN().to(device)
history_base, acc_base, recall_base = run_experiment(model_base, train_loader, val_loader, test_loader, lr=0.001, epochs=5)
plot_history(history_base, "(Baseline)")

## 3.2 Ajout d'une 3ème couche de convolution

**Question :** Est-ce que l'augmentation de la profondeur améliore significativement le score sur des images 28×28 ?

In [ ]:
class DeeperCNN(nn.Module):
    def __init__(self):
        super(DeeperCNN, self).__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2)  # 28->14
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2)  # 14->7
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2)  # 7->3
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 3 * 3, 1)
        )
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.classifier(x)
        return x

model_deep = DeeperCNN().to(device)
print(model_deep)
print(f"Paramètres: {sum(p.numel() for p in model_deep.parameters()):,}")
history_deep, acc_deep, recall_deep = run_experiment(model_deep, train_loader, val_loader, test_loader, lr=0.001, epochs=5)
plot_history(history_deep, "(3 blocs conv)")

**Analyse :** Sur des images 28×28, ajouter une 3ème couche réduit la carte à 3×3, ce qui peut provoquer une perte d'information spatiale. Le gain est souvent marginal voire négatif. La profondeur aide davantage sur des images de plus grande résolution.

## 3.3 Influence de la taille des filtres (5×5 au lieu de 3×3)

In [ ]:
class CNN_Filter5(nn.Module):
    def __init__(self):
        super(CNN_Filter5, self).__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=5, padding=2), nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=5, padding=2), nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(32*7*7, 1))
    def forward(self, x):
        return self.classifier(self.block2(self.block1(x)))

model_f5 = CNN_Filter5().to(device)
print(f"Paramètres (filtre 5x5): {sum(p.numel() for p in model_f5.parameters()):,}")
history_f5, acc_f5, recall_f5 = run_experiment(model_f5, train_loader, val_loader, test_loader, lr=0.001, epochs=5)
plot_history(history_f5, "(Filtres 5x5)")

**Analyse :** Les filtres 5×5 capturent un champ récepteur plus large à chaque couche, mais augmentent le nombre de paramètres. Sur des images 28×28, cela peut aider à capturer des structures plus larges dès la première couche.

## 3.4 Influence du nombre d'époques (15 époques)

In [ ]:
model_ep15 = SimpleCNN().to(device)
history_ep15, acc_ep15, recall_ep15 = run_experiment(model_ep15, train_loader, val_loader, test_loader, lr=0.001, epochs=15)
plot_history(history_ep15, "(15 époques)")

**Analyse :** Augmenter le nombre d'époques permet au modèle de mieux apprendre, mais au-delà d'un certain point, on risque le **surapprentissage** (overfitting) : la loss d'entraînement continue de baisser tandis que la loss de validation stagne ou augmente.

## 3.5 Influence du Batch Size

In [ ]:
results_bs = {}
for bs in [16, 32, 64, 128]:
    print(f"\n{'='*50}")
    print(f"Batch Size = {bs}")
    print('='*50)
    loader_tr = DataLoader(train_dataset, batch_size=bs, shuffle=True)
    loader_vl = DataLoader(val_dataset, batch_size=bs, shuffle=False)
    loader_ts = DataLoader(test_dataset, batch_size=bs, shuffle=False)
    m = SimpleCNN().to(device)
    h, a, r = run_experiment(m, loader_tr, loader_vl, loader_ts, lr=0.001, epochs=5)
    results_bs[bs] = {'history': h, 'acc': a, 'recall': r}

# Comparaison
fig, ax = plt.subplots(figsize=(8, 4))
for bs, res in results_bs.items():
    ax.plot(res['history']['val_acc'], label=f'BS={bs}')
ax.set_title('Accuracy de validation selon le Batch Size')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy'); ax.legend()
plt.tight_layout(); plt.show()

## 3.6 Influence du Learning Rate

In [ ]:
print("="*50)
print("LR = 0.2 (trop élevé)")
print("="*50)
model_lr_high = SimpleCNN().to(device)
h_high, a_high, r_high = run_experiment(model_lr_high, train_loader, val_loader, test_loader, lr=0.2, epochs=5)
plot_history(h_high, "(LR=0.2)")

In [ ]:
print("="*50)
print("LR = 0.00001 (trop faible)")
print("="*50)
model_lr_low = SimpleCNN().to(device)
h_low, a_low, r_low = run_experiment(model_lr_low, train_loader, val_loader, test_loader, lr=0.00001, epochs=5)
plot_history(h_low, "(LR=0.00001)")

**Réponses :**
- **LR trop élevé (0.2) :** Le modèle ne converge pas car les mises à jour des poids sont trop grandes, faisant "sauter" le minimum de la fonction de perte. La loss oscille ou diverge.
- **LR trop faible (0.00001) :** Le modèle converge très lentement. Le temps de calcul nécessaire pour atteindre de bonnes performances est considérablement augmenté, et le modèle risque de rester bloqué dans un minimum local.

## 3.7 Comparaison d'optimiseurs (Adam vs SGD vs RMSprop)

In [ ]:
optimizers = {'Adam': optim.Adam, 'SGD': optim.SGD, 'RMSprop': optim.RMSprop}
results_opt = {}
for name, opt_cls in optimizers.items():
    print(f"\n{'='*50}")
    print(f"Optimiseur: {name}")
    print('='*50)
    m = SimpleCNN().to(device)
    h, a, r = run_experiment(m, train_loader, val_loader, test_loader, lr=0.001, epochs=5, optimizer_class=opt_cls)
    results_opt[name] = {'history': h, 'acc': a, 'recall': r}

fig, ax = plt.subplots(figsize=(8, 4))
for name, res in results_opt.items():
    ax.plot(res['history']['val_acc'], label=name)
ax.set_title("Accuracy de validation par optimiseur")
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy'); ax.legend()
plt.tight_layout(); plt.show()

## 3.8 Analyse du surapprentissage (Overfitting)

**Question :** Si la précision d'entraînement atteint 99% mais la validation stagne à 85%, comment appelle-t-on ce phénomène ?

**Réponse :** C'est le **surapprentissage (overfitting)**. Le modèle a mémorisé les données d'entraînement au lieu d'apprendre des caractéristiques généralisables.

**Solutions possibles :**
1. **Dropout** : Désactiver aléatoirement des neurones pendant l'entraînement
2. **Data Augmentation** : Augmenter artificiellement la taille du dataset (rotations, flips, etc.)
3. **Régularisation L2** (weight decay) dans l'optimiseur
4. **Early Stopping** : Arrêter l'entraînement quand la validation ne s'améliore plus
5. **Réduire la complexité du modèle** (moins de paramètres)

## 3.9 Analyse médicale : Accuracy vs Recall

**Question :** Est-il plus grave de classer un patient "Sain" comme "Malade" ou un patient "Malade" comme "Sain" ?

**Réponse :** Il est **beaucoup plus grave** de classer un patient **Malade comme Sain** (faux négatif). Le patient ne recevrait pas de traitement, ce qui peut mettre sa vie en danger. Un faux positif (sain classé malade) entraîne simplement des examens complémentaires.

**Pourquoi l'Accuracy seule est insuffisante ?**
Le dataset est déséquilibré (4273 Pneumonie vs 1583 Normal). Un modèle qui prédit toujours "Pneumonie" aurait ~73% d'accuracy mais serait cliniquement inutile. Le **Recall (sensibilité)** mesure la proportion de vrais malades correctement détectés — c'est la métrique la plus importante en imagerie médicale.

### Test avec focus sur le Recall

In [ ]:
# Récapitulatif de tous les résultats
print("="*70)
print(f"{'Expérience':<30} {'Test Acc':>10} {'Test Recall':>12}")
print("="*70)
print(f"{'Baseline (2 blocs, lr=0.001)':<30} {acc_base:>10.4f} {recall_base:>12.4f}")
print(f"{'3 blocs convolution':<30} {acc_deep:>10.4f} {recall_deep:>12.4f}")
print(f"{'Filtres 5x5':<30} {acc_f5:>10.4f} {recall_f5:>12.4f}")
print(f"{'15 époques':<30} {acc_ep15:>10.4f} {recall_ep15:>12.4f}")
print(f"{'LR=0.2':<30} {a_high:>10.4f} {r_high:>12.4f}")
print(f"{'LR=0.00001':<30} {a_low:>10.4f} {r_low:>12.4f}")
for bs, res in results_bs.items():
    print(f"{'BS='+str(bs):<30} {res['acc']:>10.4f} {res['recall']:>12.4f}")
for name, res in results_opt.items():
    print(f"{name:<30} {res['acc']:>10.4f} {res['recall']:>12.4f}")
print("="*70)
print("\nEn imagerie médicale, on privilégie le modèle avec le MEILLEUR RECALL")
print("pour minimiser les faux négatifs (patients malades non détectés).")